In [8]:
import sympy as sp

# ============================================================
# Utilities
# ============================================================

def reduced_groebner_basis(gens, vars_, order='lex'):
    G = sp.groebner(gens, *vars_, order=order, domain='QQ')
    return [sp.expand(g.as_expr()) for g in G.polys]

def in_ideal(f, gens, vars_, order='lex'):
    G = sp.groebner(gens, *vars_, order=order, domain='QQ')
    _, rem = G.reduce(sp.expand(f))
    rem = sp.expand(rem)
    return rem == 0, rem

# ============================================================
# Covariance matrix Sigma
# Node order: [A,B,C,D,E,F]
# ============================================================

sAA, sAB, sAC, sAD, sAE, sAF = sp.symbols('sAA sAB sAC sAD sAE sAF')
sBB, sBC, sBD, sBE, sBF = sp.symbols('sBB sBC sBD sBE sBF')
sCC, sCD, sCE, sCF = sp.symbols('sCC sCD sCE sCF')
sDD, sDE, sDF = sp.symbols('sDD sDE sDF')
sEE, sEF = sp.symbols('sEE sEF')
sFF = sp.symbols('sFF')

Sigma = sp.Matrix([
    [sAA, sAB, sAC, sAD, sAE, sAF],
    [sAB, sBB, sBC, sBD, sBE, sBF],
    [sAC, sBC, sCC, sCD, sCE, sCF],
    [sAD, sBD, sCD, sDD, sDE, sDF],
    [sAE, sBE, sCE, sDE, sEE, sEF],
    [sAF, sBF, sCF, sDF, sEF, sFF],
])

# ============================================================
# Graph (4)
#
# A <-> B
# C <-> D
# A -> E
# B -> E
# C -> F
# D -> F
# E -> F
# ============================================================

lae, lbe, lcf, ldf, lef = sp.symbols('lae lbe lcf ldf lef')

Lambda4 = sp.Matrix([
    [0,   0,   0,   0,   lae, 0],
    [0,   0,   0,   0,   lbe, 0],
    [0,   0,   0,   0,   0,   lcf],
    [0,   0,   0,   0,   0,   ldf],
    [0,   0,   0,   0,   0,   lef],
    [0,   0,   0,   0,   0,   0],
])

psiAA, psiBB, psiCC, psiDD, psiEE, psiFF = sp.symbols(
    'psiAA psiBB psiCC psiDD psiEE psiFF'
)
psiAB, psiCD = sp.symbols('psiAB psiCD')

Psi4 = sp.Matrix([
    [psiAA, psiAB, 0,      0,      0,      0],
    [psiAB, psiBB, 0,      0,      0,      0],
    [0,      0,    psiCC, psiCD,  0,      0],
    [0,      0,    psiCD, psiDD,  0,      0],
    [0,      0,    0,      0,     psiEE,  0],
    [0,      0,    0,      0,     0,      psiFF],
])

# ============================================================
# Graph (5)
#
# Same except D -> F replaced by D <-> F
# ============================================================

lae2, lbe2, lcf2, lef2 = sp.symbols('lae2 lbe2 lcf2 lef2')

Lambda5 = sp.Matrix([
    [0,   0,   0,   0,   lae2, 0],
    [0,   0,   0,   0,   lbe2, 0],
    [0,   0,   0,   0,   0,    lcf2],
    [0,   0,   0,   0,   0,    0],
    [0,   0,   0,   0,   0,    lef2],
    [0,   0,   0,   0,   0,    0],
])

psiAA2, psiBB2, psiCC2, psiDD2, psiEE2, psiFF2 = sp.symbols(
    'psiAA2 psiBB2 psiCC2 psiDD2 psiEE2 psiFF2'
)
psiAB2, psiCD2, psiDF2 = sp.symbols('psiAB2 psiCD2 psiDF2')

Psi5 = sp.Matrix([
    [psiAA2, psiAB2, 0,       0,       0,      0],
    [psiAB2, psiBB2, 0,       0,       0,      0],
    [0,       0,     psiCC2, psiCD2,  0,      0],
    [0,       0,     psiCD2, psiDD2,  0,      psiDF2],
    [0,       0,     0,       0,      psiEE2, 0],
    [0,       0,     0,       psiDF2, 0,      psiFF2],
])

# ============================================================
# Model ideals:
#   (I - Lambda)^T Sigma (I - Lambda) - Psi = 0
# ============================================================

I = sp.eye(6)

M4 = sp.expand((I - Lambda4).T * Sigma * (I - Lambda4) - Psi4)
M5 = sp.expand((I - Lambda5).T * Sigma * (I - Lambda5) - Psi5)

# Use upper triangular entries only
J4 = []
J5 = []

for i in range(6):
    for j in range(i, 6):
        J4.append(sp.expand(M4[i, j]))
        J5.append(sp.expand(M5[i, j]))

# ============================================================
# Variable list for Groebner computations
# ============================================================

vars_all = list(Sigma.free_symbols)
vars_all += list(Lambda4.free_symbols)
vars_all += list(Psi4.free_symbols)
vars_all += list(Lambda5.free_symbols)
vars_all += list(Psi5.free_symbols)

# ============================================================
# CI polynomial for B ⟂ C | {A,D,F}
#
# rows = [B,A,D,F]
# cols = [C,A,D,F]
# ============================================================

print("Computing reduced Groebner bases...")

GB4 = reduced_groebner_basis(J4, vars_all, order='lex')
GB5 = reduced_groebner_basis(J5, vars_all, order='lex')

print("\nReduced Groebner basis of J4:")
for g in GB4:
    print(g)

print("\nReduced Groebner basis of J5:")
for g in GB5:
    print(g)

same_model_ideal = (GB4 == GB5)
print("\nDo J4 and J5 define the same ideal?")
print(same_model_ideal)

Computing reduced Groebner bases...

Reduced Groebner basis of J4:
sBD
-psiCC + sCC
-lae**2*lef**2*psiAA - 2*lae*lbe*lef**2*psiAB - lbe**2*lef**2*psiBB - lcf**2*psiCC - 2*lcf*ldf*psiCD - ldf**2*psiDD - lef**2*psiEE - psiFF + sFF
-psiAA + sAA
-psiAB + sAB
sAC
sBC
-lae*psiAB - lbe*psiBB + sBE
-psiCD + sCD
-lae**2*lef*psiAA - 2*lae*lbe*lef*psiAB - lbe**2*lef*psiBB - lef*psiEE + sEF
-psiBB + sBB
-psiDD + sDD
-lae**2*psiAA - 2*lae*lbe*psiAB - lbe**2*psiBB - psiEE + sEE
-lcf*psiCD - ldf*psiDD + sDF
sDE
sAD
-lae*lef*psiAB - lbe*lef*psiBB + sBF
-lcf*psiCC - ldf*psiCD + sCF
sCE
-lae*lef*psiAA - lbe*lef*psiAB + sAF
-lae*psiAA - lbe*psiAB + sAE

Reduced Groebner basis of J5:
sBD
-psiCC2 + sCC
-lae2**2*lef2**2*psiAA2 - 2*lae2*lbe2*lef2**2*psiAB2 - lbe2**2*lef2**2*psiBB2 - lcf2**2*psiCC2 - lef2**2*psiEE2 - psiFF2 + sFF
-psiAA2 + sAA
-psiAB2 + sAB
sAC
sBC
-lae2*psiAB2 - lbe2*psiBB2 + sBE
-psiCD2 + sCD
-lae2**2*lef2*psiAA2 - 2*lae2*lbe2*lef2*psiAB2 - lbe2**2*lef2*psiBB2 - lef2*psiEE2 + sEF
-psiBB2 + 